In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
# from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
# load MNIST data
transform = transforms.ToTensor()
train_data  = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.01MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.55MB/s]


In [5]:
# firstly see how the data look like and size of dataset
img, label = train_data[16]
print(img.size)
# plt.imshow(img)
print(f"label is {label}, and image size is {img.size}\n")
print(f"train data size {len(train_data)}, test data size {len(test_data)}")

<built-in method size of Tensor object at 0x7ea399d01310>
label is 2, and image size is <built-in method size of Tensor object at 0x7ea399d01310>

train data size 60000, test data size 10000


In [6]:
# process data get a train with 20 points, valid with 100 points rest of train are pooling points
# train and valid need random and balanced
len_train = len(train_data)
index_lst = []
for i in range(10):
  data_index = [index for index in range(len_train) if train_data[index][1] == i ]
  index_lst.append(data_index)

In [7]:
import random
train_index = []
valid_index = []
# more data for inital train set we want to train firstly let the CNN layer feature extract
for i in range(10):
  temp_choose = random.sample(index_lst[i], k=15)
  train_index.extend(temp_choose[:5])
  valid_index.extend(temp_choose[5:])
  index_lst[i] = [x for x in index_lst[i] if x not in temp_choose]


In [8]:
pooling_index = [i for j in  index_lst for i in j]
len(pooling_index)

59850

In [9]:
class NewDataset(Dataset):
  def __init__(self, index):
    imgs = []
    labels = []
    for i in index:
      img, label = train_data[i]
      imgs.append(img)
      # change label to one hot
      templabel = torch.zeros(10)
      templabel[label] = 1
      labels.append(templabel)
    self.imgs = torch.stack(imgs)
    self.labels = torch.stack(labels)

  def __len__(self):
    return len(self.imgs)

  def __getitem__(self, idx):
    return self.imgs[idx], self.labels[idx]



In [10]:
class BaseCNN(nn.Module):
  def __init__(self) -> None:
      super().__init__()
      self.covn1 =nn.Conv2d(in_channels=1, out_channels=32,  kernel_size=4)
      self.covn2 = nn.Conv2d(in_channels=32, out_channels=32,  kernel_size=4)
      self.max_pool = nn.MaxPool2d(kernel_size=2)
      self.dropout_layer1 = nn.Dropout(p=0.25)
      self.flatten = nn.Flatten()
      self.dense_layer1 = nn.Linear(in_features=3872, out_features=128)
      self.dropout_layer2 = nn.Dropout(p=0.5)
      self.dense_layer2 = nn.Linear(in_features=128, out_features=10)

  def forward(self, x, feature=False):
      x = F.relu(self.covn1(x))
      x = F.relu(self.covn2(x))
      x = self.max_pool(x)
      x = self.dropout_layer1(x)
      x = self.flatten(x)
      x = F.relu(self.dense_layer1(x))
      x = self.dropout_layer2(x)
      # print(x.shape)
      if feature:
          return x
      x = self.dense_layer2(x)
      return x

In [11]:
# train model  mainly for extract feature
def pretrain_model(trainData):
    train_loader  = DataLoader(trainData, batch_size=128, shuffle=True)
    model = BaseCNN().to(device)
    criterion = nn.MSELoss()
    weight_decay = 0.02/(len(trainData))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    # Training loop
    for epoch in tqdm.tqdm(range(50)):
      total_loss = 0
      for i, (imgs, labels) in (enumerate(train_loader)):
        optimizer.zero_grad()
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
      avg_loss = total_loss / len(train_loader)
      # if epoch % 79 == 0:
      #   print(f"/n epoch{epoch}, train_loss is {avg_loss:.4f}.")

    return model



In [12]:
# train model at first
trainData = NewDataset(index=train_index)
model = pretrain_model(trainData)

100%|██████████| 50/50 [00:01<00:00, 37.88it/s]


In [13]:
# calculate test accuracy change to report RMSE
# # calculate test accuracy


In [14]:
def predict_var(model, pooling_index, U, sigma_like):
    poolingData = NewDataset(index=pooling_index)
    pooling_loader = DataLoader(poolingData, batch_size=128, shuffle=False)
    all_var = []
    for imgs,  labels in pooling_loader:
        batch_size = imgs.size(0) #B
        phi_x = model(imgs.to(device), feature=True) #  BxK
        # temp = phi_x @ U @phi_x.t() #NK KK KN
        curr_var = sigma_like + torch.sum((phi_x @ U) * phi_x, dim=1)
        all_var.append(curr_var)
    all_var = torch.cat(all_var, dim=0)
    top_k_value, top_k_idx = torch.topk(all_var, k=10, dim=0)
    new_data_index = []
    for i in top_k_idx:
      new_data_index.append(pooling_index[i])
    # print(f"set is {set(top_k_idx.tolist())}")
    new_pooling_index  = [v for i, v in enumerate(pooling_index) if i not in set(top_k_idx.tolist()) ]
    return  new_data_index, new_pooling_index

In [25]:
# one experiment
#return pooling_index new train_index
def update_W(trainloader, sigma_likelihood, s_prior):
    feature_lst = []
    all_label = []
    for img, label in trainloader:
        img = img.to(device)
        label = label.to(device)
        feature = model(img, feature=True)
        feature_lst.append(feature)
        all_label.append(label)
    phi = torch.cat(feature_lst, dim=0) # NxK
    Y = torch.cat(all_label, dim=0) # NxD
    U = torch.inverse(phi.t()@phi + (sigma_likelihood/s_prior) * torch.eye(128).to(device)) # KxN NxK -> KxK
    M = U @ (phi.t() @ Y) # KxK KxN NxD  -> KxD correct dimension :)
    with torch.no_grad():
        model.dense_layer2.weight.copy_(M.t())
        model.dense_layer2.bias.fill_(0) # by definition of lecture note
    return  U # var

def predict_var(model, pooling_index, U, sigma_like):
    poolingData = NewDataset(index=pooling_index)
    pooling_loader = DataLoader(poolingData, batch_size=128, shuffle=False)
    all_var = []
    for imgs,  labels in pooling_loader:
        batch_size = imgs.size(0) #B
        phi_x = model(imgs.to(device), feature=True) #  BxK
        # temp = phi_x @ U @phi_x.t() #NK KK KN
        curr_var = sigma_like + torch.sum((phi_x @ U) * phi_x, dim=1)
        all_var.append(curr_var)
    all_var = torch.cat(all_var, dim=0)
    top_k_value, top_k_idx = torch.topk(all_var, k=10, dim=0)
    new_data_index = []
    for i in top_k_idx:
      new_data_index.append(pooling_index[i])
    # print(f"set is {set(top_k_idx.tolist())}")
    new_pooling_index  = [v for i, v in enumerate(pooling_index) if i not in set(top_k_idx.tolist()) ]
    return  new_data_index, new_pooling_index


def test_accuracy( model, test_data, device=device):
  test_loader = DataLoader(test_data, batch_size=500, shuffle=False)
  model.eval()

  rmse = 0
  n_data = len(test_data)
  with torch.no_grad():
    for imgs, labels in test_loader:
      imgs = imgs.to(device)
      labels = labels.to(device)
      outputs = model(imgs)
      labels_one_hot = torch.nn.functional.one_hot(labels, num_classes=10).float()
      # predicted = outputs.argmax(dim=1)
      rmse += ((outputs - labels_one_hot)**2).sum().item()
      # correct += (predicted ==labels).sum().item()
  rmse = (rmse/n_data)**0.5
  # print(f"rmse is {rmse}")
  return rmse

model.eval()
def run_once(train_index, pooling_index, test_data=test_data):
    sigma_like = 0.01
    prior_s = 1
    model.eval()
    print(f"curr size of train_data {len(train_index)}, curr size of pooling data {len(pooling_index)}  ")
    trainData = NewDataset(index=train_index)
    trainloader = DataLoader(trainData, batch_size=128, shuffle=False)
    # vaildData = NewDataset(index=valid_index)
    poolingData =  NewDataset(index=pooling_index)
    # model = train_model(trainData=trainData)
    U =update_W(trainloader, sigma_likelihood=sigma_like, s_prior=prior_s)
    # print("come here ")
    new_trainData_index, new_pool_index = predict_var( model, pooling_index, U, sigma_like)
    train_index.extend(new_trainData_index)
    test_ac = test_accuracy(model, test_data)
    print(f"test error is {test_ac}")
    return train_index, new_pool_index, test_ac


In [26]:
# number of experiement
pooling_index_temp = pooling_index.copy()
train_index_temp = train_index.copy()
n_experiement = 100
test_accuracy_lst = []
for i in range(n_experiement):
    train_index_temp, pooling_index_temp, test_ac= run_once(train_index_temp, pooling_index_temp)
    test_accuracy_lst.append(test_ac)



curr size of train_data 50, curr size of pooling data 59850  
test error is 0.7020924666930343
curr size of train_data 60, curr size of pooling data 59840  
test error is 0.7146882066944199
curr size of train_data 70, curr size of pooling data 59830  
test error is 0.7390892342052914
curr size of train_data 80, curr size of pooling data 59820  
test error is 0.7485113685167568
curr size of train_data 90, curr size of pooling data 59810  
test error is 0.7656099159467542
curr size of train_data 100, curr size of pooling data 59800  
test error is 0.7559860348900174
curr size of train_data 110, curr size of pooling data 59790  
test error is 0.7376025309008292
curr size of train_data 120, curr size of pooling data 59780  
test error is 0.7387388204642016
curr size of train_data 130, curr size of pooling data 59770  
test error is 0.7368444559045455
curr size of train_data 140, curr size of pooling data 59760  
test error is 0.7267265040874483
curr size of train_data 150, curr size of poo

In [23]:
def save_accuracy(file_name, accuracy_lst):
  path = '/content/drive/MyDrive/UDL_DATA/'
  path = path + file_name
  with open(path, 'w') as f:
      for item in accuracy_lst:
          f.write(f"{item}\n")



In [27]:
save_accuracy("analytic_extension_0.01_sigma.txt",test_accuracy_lst )